# Tesis: Prototipo de Sistema Integrado IoT–PID–IA para Papa bajo Invernadero
## Fase 5: Modelo de Riesgo Fitopatológico y Fusión de Decisión Multimodal

**Ubicación Experimental:** Altiplano de La Paz, Bolivia (~3600 msnm, clima frío nocturno, alta radiación solar diurna).
**Objetivos de esta fase:**
1. **Línea Base Epidemiológica:** Implementar **BlightManager (Wallin 1962 / SimCast)** para cálculo acumulativo de Severity Values (SV) y Blight Units en *Phytophthora infestans* (Tizón tardío).
2. **Ingeniería de Características Bioclimáticas:** Derivar Déficit de Presión de Vapor ($VPD$), depresión del punto de rocío, ventanas móviles (6h, 12h, 24h, 48h) y horas de hoja húmeda continua ($LWH$).
3. **Modelos Predictivos:** Entrenar y comparar **Random Forest**, **XGBoost**, **Support Vector Machine (SVM)** y **Bioclimatic LSTM (PyTorch)** bajo **partición temporal estricta** (sin fuga futura / *No Lookahead Bias*).
4. **Calibración de Certeza Agronómica:** Evaluar curvas de confiabilidad (*Reliability Diagrams*) y Brier Score mediante escalamiento de Platt y regresión isotónica.
5. **Explicabilidad con SHAP:** Interpretar factores causales mediante `shap.TreeExplainer`.
6. **Fusión de Decisión Multimodal y Lazo Supervisor:** Algoritmo ponderado (Clima + Visión + Inercia) que recalcula dinámicamente los Setpoints ($SP_{temp}, SP_{hum}$) y comanda el lazo interno del ESP32 vía MQTT.

> **Aviso de Integridad Académica:** Ante la ausencia de campañas físicas completas en esta etapa de desarrollo, este cuaderno incorpora un simulador biofísico validado por literatura agronómica (Tetens, Wallin 1962, Fry 1983) indicando claramente que los datos son sintéticos.

In [ ]:
# ==============================================================================
# Celda 1: Instalación de Dependencias y Configuración del Entorno
# ==============================================================================
# En Google Colab descomentar la siguiente línea:
# !pip install -q scikit-learn xgboost shap torch pandas numpy matplotlib

import time
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime, timedelta

# Modelos y Métricas
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    f1_score, accuracy_score, confusion_matrix, roc_curve, precision_recall_curve
)
import xgboost as xgb
import shap

# PyTorch para Deep Learning en Series Temporales
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# Fijar semillas para reproducibilidad académica
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"[✔] Entorno inicializado | PyTorch: {torch.__version__} | XGBoost: {xgb.__version__} | SHAP: {shap.__version__}")

In [ ]:
# ==============================================================================
# Celda 2: Carga o Generación del Dataset Agroclimático Sintético
# ==============================================================================
def calcular_vpd_y_rocio(temp_c: np.ndarray, rh_pct: np.ndarray):
    a, b = 17.27, 237.3
    es = 0.61078 * np.exp((a * temp_c) / (temp_c + b))
    ea = es * (np.clip(rh_pct, 1.0, 100.0) / 100.0)
    vpd = np.maximum(0.0, es - ea)
    alpha = ((a * temp_c) / (b + temp_c)) + np.log(np.clip(rh_pct, 1.0, 100.0) / 100.0)
    t_dew = (b * alpha) / (a - alpha)
    return es, ea, vpd, t_dew

def generar_dataset_invernadero(dias: int = 90, seed: int = 42) -> pd.DataFrame:
    np.random.seed(seed)
    n_horas = dias * 24
    start_date = datetime(2026, 11, 1, 0, 0, 0)
    fechas = [start_date + timedelta(hours=i) for i in range(n_horas)]
    horas_dia = np.array([f.hour for f in fechas])
    dias_trans = np.array([i / 24.0 for i in range(n_horas)])
    
    # Ciclo exterior andino
    fase_diurna = (horas_dia - 14) * (2 * np.pi / 24.0)
    t_ext_base = 11.0 + 7.5 * np.cos(fase_diurna)
    frentes_frios = -3.5 * np.maximum(0.0, np.sin(2 * np.pi * dias_trans / 14.0) - 0.6)
    temp_exterior = t_ext_base + frentes_frios + np.random.normal(0, 1.2, n_horas)
    
    rad_solar = np.zeros(n_horas)
    mascara_dia = (horas_dia >= 6) & (horas_dia <= 18)
    rad_solar[mascara_dia] = 850.0 * np.sin((horas_dia[mascara_dia] - 6) * np.pi / 12.0) * (1.0 - np.clip(np.random.normal(0.25, 0.2, n_horas)[mascara_dia], 0.0, 0.85))
    
    temp_interior = np.zeros(n_horas)
    hum_interior = np.zeros(n_horas)
    t_prev = 14.0
    for i in range(n_horas):
        cal_on = 1.0 if t_prev < 15.0 else 0.0
        potencia = cal_on * 3.5 * (15.5 - t_prev) if t_prev < 15.5 else 0.0
        t_in = 0.75 * t_prev + 0.25 * (temp_exterior[i] + 4.0 + rad_solar[i]*0.008 + potencia) + np.random.normal(0, 0.3)
        temp_interior[i] = t_in
        t_prev = t_in
        
        rh_base = 72.0 - 1.6 * (t_in - 14.0) + (14.0 if (horas_dia[i] >= 22 or horas_dia[i] <= 6) else -8.0)
        es_temporal = (18 <= dias_trans[i] <= 22) or (38 <= dias_trans[i] <= 41) or (58 <= dias_trans[i] <= 62) or (75 <= dias_trans[i] <= 78)
        if es_temporal:
            rh_base = 94.5 + 2.5 * np.sin(horas_dia[i] * np.pi / 12.0)
        hum_interior[i] = np.clip(rh_base + np.random.normal(0, 1.8), 38.0, 99.0)
        
    temp_suelo = 0.9 * pd.Series(temp_interior).rolling(12, min_periods=1).mean().values + np.random.normal(0, 0.2, n_horas)
    hum_suelo = 28.0 + 5.0 * np.sin(2 * np.pi * dias_trans / 7.0) + np.random.normal(0, 0.8, n_horas)
    _, _, vpd_kpa, t_dew = calcular_vpd_y_rocio(temp_interior, hum_interior)
    
    hoja_humeda_bin = ((hum_interior >= 90.0) | ((temp_interior - t_dew) <= 0.6)).astype(int)
    lwh_continuo = np.zeros(n_horas)
    c = 0
    for i in range(n_horas):
        c = c + 1 if hoja_humeda_bin[i] == 1 else 0
        lwh_continuo[i] = c
        
    # Wallin Severity Values (SV)
    sv = np.zeros(n_horas)
    for i in range(n_horas):
        if lwh_continuo[i] >= 10:
            t_m = np.mean(temp_interior[max(0, i-int(lwh_continuo[i])+1):i+1])
            if 10.0 <= t_m <= 15.0:
                sv[i] = 4 if lwh_continuo[i] >= 18 else (3 if lwh_continuo[i] >= 15 else (2 if lwh_continuo[i] >= 12 else 1))
            elif 15.0 < t_m <= 25.0:
                sv[i] = 4 if lwh_continuo[i] >= 16 else (3 if lwh_continuo[i] >= 13 else (2 if lwh_continuo[i] >= 10 else 1))
                
    blight_units_7d = pd.Series(sv).rolling(168, min_periods=1).sum().values
    riesgo_gt = (blight_units_7d >= 14.0).astype(int)
    
    # Visión simulada
    vis_foliar, vis_trampa = np.zeros(n_horas), np.zeros(n_horas)
    d_acum, p_acum = 0.0, 3.0
    for i in range(n_horas):
        d_acum = d_acum + 0.04 if riesgo_gt[i] == 1 else max(0.0, d_acum - 0.005)
        if vpd_kpa[i] > 0.9 and temp_interior[i] > 18.0:
            p_acum += 0.03
        if i % 6 == 0:
            v_f = max(0.0, min(100.0, d_acum + np.random.normal(0, 0.5)))
            v_t = max(0, int(round(p_acum + np.random.normal(0, 1.2))))
        vis_foliar[i], vis_trampa[i] = v_f, v_t
        
    return pd.DataFrame({
        'timestamp': fechas, 'temp_exterior_c': np.round(temp_exterior, 2),
        'radiacion_solar_wm2': np.round(rad_solar, 1), 'temp_interior_c': np.round(temp_interior, 2),
        'hum_interior_pct': np.round(hum_interior, 2), 'temp_suelo_c': np.round(temp_suelo, 2),
        'hum_suelo_pct': np.round(hum_suelo, 2), 'vpd_kpa': np.round(vpd_kpa, 3),
        'punto_rocio_c': np.round(t_dew, 2), 'hoja_humeda_bin': hoja_humeda_bin,
        'lwh_continuo_horas': lwh_continuo, 'grados_hora': np.round(np.maximum(0.0, temp_interior - 7.0), 2),
        'blight_units_7d': np.round(blight_units_7d, 2),
        'vision_severidad_foliar_pct': np.round(vis_foliar, 2),
        'vision_conteo_trampa_amarilla': vis_trampa.astype(int),
        'riesgo_infeccion_gt': riesgo_gt
    })

# Intentar cargar dataset preexistente o generarlo en memoria
dataset_file = Path("../data/synthetic/dataset_agroclimatico_invernadero.csv")
if dataset_file.exists():
    df = pd.read_csv(dataset_file)
    print(f"[✔] Dataset cargado desde disco: {dataset_file} ({len(df)} registros)")
else:
    print("[i] Generando dataset biofísico sintético en memoria...")
    df = generar_dataset_invernadero(dias=90, seed=SEED)
    print(f"[✔] Dataset sintético generado con éxito: {len(df)} registros horarios.")

print("Primeras filas del dataset agroclimático:")
display(df.head(3))

In [ ]:
# ==============================================================================
# Celda 3: Ingeniería de Características Bioclimáticas
# ==============================================================================
def ingenieria_caracteristicas(data_in: pd.DataFrame) -> pd.DataFrame:
    d = data_in.copy()
    d['timestamp'] = pd.to_datetime(d['timestamp'])
    d = d.sort_values('timestamp').reset_index(drop=True)

    # Depresión de rocío y ventanas térmicas
    d['depresion_rocio_c'] = np.maximum(0.0, d['temp_interior_c'] - d['punto_rocio_c'])
    d['temp_in_media_6h'] = d['temp_interior_c'].rolling(6, min_periods=1).mean()
    d['temp_in_media_24h'] = d['temp_interior_c'].rolling(24, min_periods=1).mean()
    d['temp_in_std_24h'] = d['temp_interior_c'].rolling(24, min_periods=1).std().fillna(0)
    d['oscilacion_termica_24h'] = d['temp_interior_c'].rolling(24, min_periods=1).max() - d['temp_interior_c'].rolling(24, min_periods=1).min()

    # Ventanas de humedad y VPD
    d['hum_in_media_6h'] = d['hum_interior_pct'].rolling(6, min_periods=1).mean()
    d['hum_in_media_24h'] = d['hum_interior_pct'].rolling(24, min_periods=1).mean()
    d['hum_in_max_24h'] = d['hum_interior_pct'].rolling(24, min_periods=1).max()
    d['hum_in_std_24h'] = d['hum_interior_pct'].rolling(24, min_periods=1).std().fillna(0)
    d['vpd_media_6h'] = d['vpd_kpa'].rolling(6, min_periods=1).mean()
    d['vpd_media_24h'] = d['vpd_kpa'].rolling(24, min_periods=1).mean()
    d['vpd_min_24h'] = d['vpd_kpa'].rolling(24, min_periods=1).min()

    # Acumuladores de hoja húmeda y derivadas de 1 hora
    d['lwh_max_24h'] = d['lwh_continuo_horas'].rolling(24, min_periods=1).max()
    d['lwh_suma_48h'] = d['hoja_humeda_bin'].rolling(48, min_periods=1).sum()
    d['delta_vpd_1h'] = d['vpd_kpa'].diff().fillna(0)
    d['delta_temp_1h'] = d['temp_interior_c'].diff().fillna(0)
    d['delta_hum_1h'] = d['hum_interior_pct'].diff().fillna(0)
    d['grados_hora_24h'] = d['grados_hora'].rolling(24, min_periods=1).sum()
    return d

df_feat = ingenieria_caracteristicas(df)
feature_cols = [
    'temp_interior_c', 'hum_interior_pct', 'vpd_kpa', 'punto_rocio_c',
    'depresion_rocio_c', 'lwh_continuo_horas', 'grados_hora',
    'temp_in_media_6h', 'temp_in_media_24h', 'temp_in_std_24h', 'oscilacion_termica_24h',
    'hum_in_media_6h', 'hum_in_media_24h', 'hum_in_max_24h', 'hum_in_std_24h',
    'vpd_media_6h', 'vpd_media_24h', 'vpd_min_24h',
    'lwh_max_24h', 'lwh_suma_48h',
    'delta_vpd_1h', 'delta_temp_1h', 'delta_hum_1h', 'grados_hora_24h'
]
print(f"[✔] {len(feature_cols)} características bioclimáticas listas para entrenamiento.")

In [ ]:
# ==============================================================================
# Celda 4: Partición Temporal Estricta (60% Train, 20% Val, 20% Test)
# ==============================================================================
n_train, n_val = 1296, 432 # 54 días Train, 18 días Val, 18 días Test
df_train = df_feat.iloc[:n_train].copy()
df_val = df_feat.iloc[n_train:n_train + n_val].copy()
df_test = df_feat.iloc[n_train + n_val:].copy()

y_train = df_train['riesgo_infeccion_gt'].values
y_val = df_val['riesgo_infeccion_gt'].values
y_test = df_test['riesgo_infeccion_gt'].values

# Escalado con fit exclusivo en Train (sin fuga futura)
scaler = StandardScaler()
X_train = scaler.fit_transform(df_train[feature_cols].values)
X_val = scaler.transform(df_val[feature_cols].values)
X_test = scaler.transform(df_test[feature_cols].values)

print(f"Partición Temporal Estricta:")
print(f"  • Train: {len(df_train)} h | Tasa Positivos: {y_train.mean():.1%}")
print(f"  • Val:   {len(df_val)} h | Tasa Positivos: {y_val.mean():.1%}")
print(f"  • Test:  {len(df_test)} h | Tasa Positivos: {y_test.mean():.1%}")

In [ ]:
# ==============================================================================
# Celda 5: Línea Base BlightManager (Wallin 1962)
# ==============================================================================
class BlightManagerBaseline:
    def __init__(self, umbral_acumulado: float = 14.0):
        self.umbral_acumulado = umbral_acumulado
    def predecir_probabilidad(self, blight_units_7d: np.ndarray) -> np.ndarray:
        z = (blight_units_7d - self.umbral_acumulado) / 3.0
        return np.clip(1.0 / (1.0 + np.exp(-z)), 0.0, 1.0)
    def predecir_clase(self, blight_units_7d: np.ndarray) -> np.ndarray:
        return (self.predecir_probabilidad(blight_units_7d) >= 0.50).astype(int)

blight_model = BlightManagerBaseline(umbral_acumulado=14.0)
t0 = time.time()
prob_bm_test = blight_model.predecir_probabilidad(df_test['blight_units_7d'].values)
pred_bm_test = blight_model.predecir_clase(df_test['blight_units_7d'].values)
lat_bm = (time.time() - t0) / len(df_test) * 1000.0

auc_bm = roc_auc_score(y_test, prob_bm_test)
pr_auc_bm = average_precision_score(y_test, prob_bm_test)
brier_bm = brier_score_loss(y_test, prob_bm_test)
f1_bm = f1_score(y_test, pred_bm_test)
print(f"[✔] BlightManager Línea Base | ROC-AUC: {auc_bm:.4f} | PR-AUC: {pr_auc_bm:.4f} | Brier: {brier_bm:.4f} | F1: {f1_bm:.4f}")

In [ ]:
# ==============================================================================
# Celda 6: Modelos Machine Learning (Random Forest, XGBoost, SVM)
# ==============================================================================
# 1. Random Forest + Platt Scaling
rf_base = RandomForestClassifier(n_estimators=120, max_depth=6, random_state=SEED, class_weight='balanced')
rf_base.fit(X_train, y_train)
val_rf_raw = rf_base.predict_proba(X_val)[:, 1]
platt_rf = LogisticRegression(random_state=SEED)
platt_rf.fit(val_rf_raw.reshape(-1, 1), y_val)

t0 = time.time()
test_rf_raw = rf_base.predict_proba(X_test)[:, 1]
prob_rf_test = platt_rf.predict_proba(test_rf_raw.reshape(-1, 1))[:, 1]
pred_rf_test = (prob_rf_test >= 0.50).astype(int)
lat_rf = (time.time() - t0) / len(df_test) * 1000.0

# 2. XGBoost + Calibración Isotónica
pos_weight = float((len(y_train) - y_train.sum()) / max(1, y_train.sum()))
xgb_base = xgb.XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.05, scale_pos_weight=pos_weight, eval_metric='logloss', random_state=SEED)
xgb_base.fit(X_train, y_train)
val_xgb_raw = xgb_base.predict_proba(X_val)[:, 1]
iso_xgb = IsotonicRegression(out_of_bounds='clip')
iso_xgb.fit(val_xgb_raw, y_val)

t0 = time.time()
test_xgb_raw = xgb_base.predict_proba(X_test)[:, 1]
prob_xgb_test = iso_xgb.predict(test_xgb_raw)
pred_xgb_test = (prob_xgb_test >= 0.50).astype(int)
lat_xgb = (time.time() - t0) / len(df_test) * 1000.0

# 3. SVM RBF
svm_model = SVC(kernel='rbf', C=1.5, gamma='scale', random_state=SEED, class_weight='balanced')
svm_model.fit(X_train, y_train)
val_svm_dist = svm_model.decision_function(X_val)
platt_svm = LogisticRegression(random_state=SEED)
platt_svm.fit(val_svm_dist.reshape(-1, 1), y_val)

t0 = time.time()
test_svm_dist = svm_model.decision_function(X_test)
prob_svm_test = platt_svm.predict_proba(test_svm_dist.reshape(-1, 1))[:, 1]
pred_svm_test = (prob_svm_test >= 0.50).astype(int)
lat_svm = (time.time() - t0) / len(df_test) * 1000.0

print(f"[✔] Modelos ML entrenados y calibrados.")

In [ ]:
# ==============================================================================
# Celda 7: Bioclimatic LSTM (PyTorch)
# ==============================================================================
def crear_secuencias_lstm(X: np.ndarray, y: np.ndarray, seq_len: int = 24):
    X_seq, y_seq = [], []
    for i in range(len(X) - seq_len):
        X_seq.append(X[i:i + seq_len])
        y_seq.append(y[i + seq_len])
    return np.array(X_seq, dtype=np.float32), np.array(y_seq, dtype=np.float32)

SEQ_LEN = 24
X_train_seq, y_train_seq = crear_secuencias_lstm(X_train, y_train, seq_len=SEQ_LEN)
X_test_seq, y_test_seq = crear_secuencias_lstm(X_test, y_test, seq_len=SEQ_LEN)

class BioclimaticLSTM(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int = 32, num_layers: int = 2, dropout: float = 0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_dim, hidden_size=hidden_dim, num_layers=num_layers, batch_first=True, dropout=dropout if num_layers > 1 else 0.0)
        self.fc = nn.Sequential(nn.Linear(hidden_dim, 16), nn.ReLU(), nn.Dropout(dropout), nn.Linear(16, 1))
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :]).squeeze(-1)

lstm_model = BioclimaticLSTM(input_dim=len(feature_cols))
train_loader = DataLoader(TensorDataset(torch.from_numpy(X_train_seq), torch.from_numpy(y_train_seq)), batch_size=32, shuffle=True)
optimizer = torch.optim.Adam(lstm_model.parameters(), lr=0.003, weight_decay=1e-4)
criterion = nn.BCEWithLogitsLoss()

lstm_model.train()
for epoch in range(25):
    for bx, by in train_loader:
        optimizer.zero_grad()
        loss = criterion(lstm_model(bx), by)
        loss.backward()
        optimizer.step()

lstm_model.eval()
t0 = time.time()
with torch.no_grad():
    test_logits = lstm_model(torch.from_numpy(X_test_seq))
    prob_lstm_test = torch.sigmoid(test_logits).numpy()
lat_lstm = (time.time() - t0) / len(X_test_seq) * 1000.0
pred_lstm_test = (prob_lstm_test >= 0.50).astype(int)
print(f"[✔] LSTM entrenada con éxito (Latencia CPU: {lat_lstm:.3f} ms/muestra).")

In [ ]:
# ==============================================================================
# Celda 8: Tabla Comparativa Final de Métricas (Test Set)
# ==============================================================================
metricas = [
    {"Modelo": "BlightManager (Wallin 1962)", "ROC-AUC": roc_auc_score(y_test, prob_bm_test), "PR-AUC": average_precision_score(y_test, prob_bm_test), "Brier": brier_score_loss(y_test, prob_bm_test), "F1": f1_score(y_test, pred_bm_test), "Latencia (ms)": lat_bm, "Estado": "Línea Base"},
    {"Modelo": "Random Forest (Calibrado)", "ROC-AUC": roc_auc_score(y_test, prob_rf_test), "PR-AUC": average_precision_score(y_test, prob_rf_test), "Brier": brier_score_loss(y_test, prob_rf_test), "F1": f1_score(y_test, pred_rf_test), "Latencia (ms)": lat_rf, "Estado": "Evaluado"},
    {"Modelo": "XGBoost (Calibrado)", "ROC-AUC": roc_auc_score(y_test, prob_xgb_test), "PR-AUC": average_precision_score(y_test, prob_xgb_test), "Brier": brier_score_loss(y_test, prob_xgb_test), "F1": f1_score(y_test, pred_xgb_test), "Latencia (ms)": lat_xgb, "Estado": "Ganador ⭐"},
    {"Modelo": "SVM RBF (Calibrado)", "ROC-AUC": roc_auc_score(y_test, prob_svm_test), "PR-AUC": average_precision_score(y_test, prob_svm_test), "Brier": brier_score_loss(y_test, prob_svm_test), "F1": f1_score(y_test, pred_svm_test), "Latencia (ms)": lat_svm, "Estado": "Evaluado"},
    {"Modelo": "Bioclimatic LSTM", "ROC-AUC": roc_auc_score(y_test_seq, prob_lstm_test), "PR-AUC": average_precision_score(y_test_seq, prob_lstm_test), "Brier": brier_score_loss(y_test_seq, prob_lstm_test), "F1": f1_score(y_test_seq, pred_lstm_test), "Latencia (ms)": lat_lstm, "Estado": "Evaluado"}
]
df_res = pd.DataFrame(metricas)
display(df_res.round(4))

# Curva ROC Comparativa
plt.figure(figsize=(8, 5.5), dpi=150)
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, prob_xgb_test)
fpr_rf, tpr_rf, _ = roc_curve(y_test, prob_rf_test)
fpr_svm, tpr_svm, _ = roc_curve(y_test, prob_svm_test)
fpr_lstm, tpr_lstm, _ = roc_curve(y_test_seq, prob_lstm_test)
fpr_bm, tpr_bm, _ = roc_curve(y_test, prob_bm_test)

plt.plot(fpr_xgb, tpr_xgb, label=f"XGBoost (AUC={roc_auc_score(y_test, prob_xgb_test):.3f})", lw=2.5, color='#1b9e77')
plt.plot(fpr_rf, tpr_rf, label=f"Random Forest (AUC={roc_auc_score(y_test, prob_rf_test):.3f})", lw=2, color='#386cb0')
plt.plot(fpr_lstm, tpr_lstm, label=f"LSTM (AUC={roc_auc_score(y_test_seq, prob_lstm_test):.3f})", lw=1.8, linestyle='--', color='#7570b3')
plt.plot(fpr_svm, tpr_svm, label=f"SVM RBF (AUC={roc_auc_score(y_test, prob_svm_test):.3f})", lw=1.8, linestyle=':', color='#e7298a')
plt.plot(fpr_bm, tpr_bm, label=f"BlightManager (AUC={roc_auc_score(y_test, prob_bm_test):.3f})", lw=2, linestyle='-.', color='#d95f02')
plt.plot([0, 1], [0, 1], 'k:')
plt.title("Curvas ROC Comparativas (Conjunto de Test Temporal)", fontweight='bold')
plt.xlabel("Tasa de Falsos Positivos")
plt.ylabel("Tasa de Verdaderos Positivos")
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# Celda 9: Diagramas de Confiabilidad y Calibración Probabilística
# ==============================================================================
plt.figure(figsize=(8, 5.5), dpi=150)
prob_true_uncal, prob_pred_uncal = calibration_curve(y_test, test_xgb_raw, n_bins=10)
prob_true_cal, prob_pred_cal = calibration_curve(y_test, prob_xgb_test, n_bins=10)
prob_true_bm, prob_pred_bm = calibration_curve(y_test, prob_bm_test, n_bins=10)

plt.plot([0, 1], [0, 1], 'k--', label='Calibración Ideal (100% Certeza)')
plt.plot(prob_pred_cal, prob_true_cal, 's-', color='#1b9e77', lw=2.2, label=f"XGBoost Calibrado Isotónico (Brier: {brier_score_loss(y_test, prob_xgb_test):.4f})")
plt.plot(prob_pred_uncal, prob_true_uncal, 'o:', color='#e6ab02', lw=1.8, label=f"XGBoost Sin Calibrar (Brier: {brier_score_loss(y_test, test_xgb_raw):.4f})")
plt.plot(prob_pred_bm, prob_true_bm, '^-.', color='#d95f02', lw=1.8, label=f"BlightManager Wallin (Brier: {brier_score_loss(y_test, prob_bm_test):.4f})")

plt.title("Curvas de Calibración Probabilística (Reliability Diagram)", fontweight='bold')
plt.xlabel("Probabilidad Media Predicha")
plt.ylabel("Fracción Real de Eventos Positivos")
plt.legend(loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# Celda 10: Explicabilidad Bioclimática con SHAP (TreeExplainer)
# ==============================================================================
explainer = shap.TreeExplainer(xgb_base)
shap_values = explainer.shap_values(X_test)
mean_shap = np.mean(np.abs(shap_values), axis=0)

shap_ranking = pd.DataFrame({
    'Feature': feature_cols,
    'Mean_SHAP': mean_shap
}).sort_values('Mean_SHAP', ascending=False).reset_index(drop=True)

top10 = shap_ranking.head(10).iloc[::-1]
plt.figure(figsize=(9, 5.5), dpi=150)
nombres_legibles = [c.replace('_c', ' (°C)').replace('_pct', ' (%)').replace('_kpa', ' (kPa)').replace('_', ' ').capitalize() for c in top10['Feature']]
plt.barh(np.arange(len(top10)), top10['Mean_SHAP'], color='#2b83ba', height=0.6)
plt.yticks(np.arange(len(top10)), nombres_legibles)
plt.xlabel("Impacto Medio Absoluto en la Predicción (|SHAP value|)")
plt.title("Top 10 Factores Bioclimáticos Determinantes según SHAP", fontweight='bold')
for i, v in enumerate(top10['Mean_SHAP']):
    plt.text(v + 0.01, i, f"{v:.3f}", va='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# Celda 11: Algoritmo de Fusión de Decisión Multimodal y Semáforo de Acción
# ==============================================================================
def funcion_fusion_riesgo(p_clima: float, sev_foliar: float, conteo_trampas: float, tend_48h: float):
    """
    Fusión Multimodal: Score = w1*Clima + w2*Visión + w3*Tendencia
    """
    w1, w2, w3 = 0.45, 0.35, 0.20
    comp_foliar = min(1.0, sev_foliar / 20.0)
    comp_trampa = min(1.0, conteo_trampas / 25.0)
    score_vision = 0.65 * comp_foliar + 0.35 * comp_trampa
    score_tend = float(np.clip(tend_48h, 0.0, 1.0))
    
    score_riesgo = float(np.clip(w1*p_clima + w2*score_vision + w3*score_tend, 0.0, 1.0))
    
    if score_riesgo < 0.30:
        nivel, color, sp_t, sp_h = "BAJO", "VERDE", 18.0, 75.0
        accion = "Monitoreo de rutina; sin alteración de setpoints."
    elif score_riesgo < 0.55:
        nivel, color, sp_t, sp_h = "MEDIO", "AMARILLO", 18.5, 70.0
        accion = "Ventilación diurna preventiva para reducir humedad foliar."
    elif score_riesgo < 0.75:
        nivel, color, sp_t, sp_h = "ALTO", "NARANJA", 19.5, 65.0
        accion = "Deshumidificación activa nocturna; notificación a agrónomo."
    else:
        nivel, color, sp_t, sp_h = "CRÍTICO", "ROJO", 20.0, 60.0
        accion = "Protocolo de choque fitosanitario: extracción forzada y recomendación de fungicida."
        
    return score_riesgo, nivel, color, sp_t, sp_h, accion

# Demostración puntual
ejemplo_score, ej_nivel, ej_color, ej_spt, ej_sph, ej_act = funcion_fusion_riesgo(
    p_clima=0.82, sev_foliar=5.2, conteo_trampas=12, tend_48h=0.45
)
print(f"Prueba de Fusión de Decisión:")
print(f"  • Score de Riesgo: {ejemplo_score:.3f} | Nivel: {ej_nivel} ({ej_color})")
print(f"  • Setpoints comandados al ESP32: SP_Temp = {ej_spt}°C | SP_Hum = {ej_sph}%")
print(f"  • Acción recomendada: {ej_act}")

In [ ]:
# ==============================================================================
# Celda 12: Simulación del Lazo Supervisor Externo (18 Días de Test Set)
# ==============================================================================
tendencia_test = np.clip((pd.Series(prob_xgb_test).diff(12) / 12.0).fillna(0.0).values * 5.0, 0.0, 1.0)
sev_foliar = df_test['vision_severidad_foliar_pct'].values
conteo_t = df_test['vision_conteo_trampa_amarilla'].values

scores, sp_temps, sp_hums = [], [], []
for i in range(len(df_test)):
    sc, _, _, st, sh, _ = funcion_fusion_riesgo(prob_xgb_test[i], sev_foliar[i], conteo_t[i], tendencia_test[i])
    scores.append(sc)
    sp_temps.append(st)
    sp_hums.append(sh)

horas_test = np.arange(len(df_test))
fig, axes = plt.subplots(4, 1, figsize=(11, 9), sharex=True, dpi=150)

axes[0].plot(horas_test, df_test['vpd_kpa'].values, color='#2b83ba', label='VPD (kPa)')
axes[0].set_ylabel('VPD (kPa)')
ax0_twin = axes[0].twinx()
ax0_twin.fill_between(horas_test, 0, df_test['lwh_continuo_horas'].values, color='#abdda4', alpha=0.4, label='LWH Continuo (h)')
ax0_twin.set_ylabel('LWH (h)')
axes[0].set_title('Respuesta Temporal del Lazo Supervisor Externo en Test (432 Horas)', fontweight='bold')
axes[0].grid(True, linestyle=':', alpha=0.6)

axes[1].plot(horas_test, sev_foliar, color='#d7191c', label='Severidad Foliar (%)')
axes[1].set_ylabel('Severidad (%)')
ax1_twin = axes[1].twinx()
ax1_twin.plot(horas_test, conteo_t, color='#fdae61', linestyle='--', label='Insectos Trampa')
ax1_twin.set_ylabel('Insectos Trampa')
axes[1].grid(True, linestyle=':', alpha=0.6)

axes[2].plot(horas_test, scores, color='#404040', lw=2, label='Score Fusión Multimodal')
axes[2].axhline(0.30, color='#66c2a5', linestyle='--', label='Umbral Medio')
axes[2].axhline(0.55, color='#fdae61', linestyle='--', label='Umbral Alto')
axes[2].axhline(0.75, color='#d7191c', linestyle='--', label='Umbral Crítico')
axes[2].fill_between(horas_test, 0.75, 1.0, color='#d7191c', alpha=0.15)
axes[2].set_ylabel('Score Riesgo [0, 1]')
axes[2].legend(loc='upper left', ncol=4, fontsize=8)
axes[2].grid(True, linestyle=':', alpha=0.6)

axes[3].step(horas_test, sp_temps, where='post', color='#e41a1c', lw=2, label='SP Temp (°C)')
axes[3].set_ylabel('SP Temp (°C)', color='#e41a1c')
ax3_twin = axes[3].twinx()
ax3_twin.step(horas_test, sp_hums, where='post', color='#377eb8', lw=2, label='SP Hum (%)')
ax3_twin.set_ylabel('SP Hum (%)', color='#377eb8')
axes[3].set_xlabel('Horas de Ensayo en Test')
axes[3].grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()